# AML Investigation Prioritization — Guided Python Walkthrough

**Business question:** When an AML team can investigate only a fixed number of records, can a model put more relevant *synthetic benchmark labels* near the front of the queue?

This is a **step-by-step, beginner-friendly analysis** for people comfortable with pandas and basic statistics. It follows the same analytical story as the [live case study](https://mohammedmia1.github.io/AML_Data_Science_Project/).

### What you will learn

1. Inspect an account-day dataset and split it by time.
2. Turn raw feature columns into numeric model inputs.
3. Compare a simple logistic-regression baseline with XGBoost **on validation data**.
4. Translate model scores into a fixed review queue.
5. Confirm the held-out test results and understand a few **MLOps** practices.

**Important:** Every label is synthetic. A positive label does **not** establish actual crime, financial impact, or production performance. Only nine activity-based features enter the models.

## 1. Load the data and inspect its shape

**Why this matters:** Data analysts usually start by checking rows, columns, duplicates and dates. Machine learning starts the same way.

Each record represents **one account on one day**. We require the exact frozen TSV from `data/README.md`; the full file is not on GitHub because of its size.

The short setup below works in Jupyter launched from either the repository root or `notebooks/`. You can also set `AML_INPUT_DIR` to the folder containing the TSV.

In [1]:
from pathlib import Path
import hashlib
import os
import pandas as pd
import numpy as np

# Find the project folder, regardless of which of these two folders Jupyter starts in.
project = Path.cwd()
if not (project / "reports" / "metrics.json").exists():
    project = project.parent

# Identify the exact dataset used for the published model.
data_folder = Path(os.environ.get("AML_INPUT_DIR", project / "data" / "local"))
data_file = data_folder / "aml_modeling_dev.tsv"
if not data_file.exists():
    raise FileNotFoundError(f"Missing dataset: {data_file}")

with open(data_file, "rb") as source:
    file_hash = hashlib.file_digest(source, "sha256").hexdigest()

expected_hash = "d6194ac2c29f4a90e4fa1e1740945ca2f558388a799020a15f16623eee9c6012"
assert file_hash == expected_hash, "The input differs from the published dataset."

transactions = pd.read_csv(data_file, sep="\t")
transactions["activity_date"] = pd.to_datetime(transactions["activity_date"])

print("Rows:", len(transactions))
print("Columns:", len(transactions.columns))
print("Dates:", transactions["activity_date"].min().date(), "to", transactions["activity_date"].max().date())
print("Positive labels:", int(transactions["has_laundering_tx"].sum()))
print("Duplicate account-days:", transactions.duplicated(["account_id", "activity_date"]).sum())

Rows: 2479354
Columns: 13
Dates: 2022-09-01 to 2022-09-10
Positive labels: 7131


Duplicate account-days: 0


**Interpretation:** This is a large, imbalanced classification dataset. Most account-days have a zero label. A model that predicts zero for everything might have high accuracy, but it would not help investigators decide what to review.

## 2. Separate past activity from future activity

**Why this matters:** In a real analysis, we shouldn't train a model using future records and then pretend we are predicting the past. Instead of random shuffling, we divide records by date.

- **Training:** September 1–6, 2022 — learn model patterns.
- **Validation:** September 7–8 — compare methods and assess the review policy.
- **Test:** September 9–10 — assess the frozen choice on later, unseen records.

Using dates this way helps reduce **temporal leakage**. The dates and account IDs do not become model features.

In [2]:
train = transactions[transactions["activity_date"].between("2022-09-01", "2022-09-06")]
validation = transactions[transactions["activity_date"].between("2022-09-07", "2022-09-08")]
test = transactions[transactions["activity_date"].between("2022-09-09", "2022-09-10")]

split_summary = pd.DataFrame({
    "Rows": [len(train), len(validation), len(test)],
    "Positive labels": [int(train["has_laundering_tx"].sum()),
                        int(validation["has_laundering_tx"].sum()),
                        int(test["has_laundering_tx"].sum())]
}, index=["Training", "Validation", "Test"])

print(split_summary.to_string())
assert list(split_summary["Rows"]) == [1534403, 450244, 494707]

               Rows  Positive labels
Training    1534403             4003
Validation   450244             1629
Test         494707             1499


## 3. Build nine easy-to-explain features

A **feature** is simply a column used to make a prediction. Our nine features describe transaction volume, money movement, banking diversity and rapid activity.

Some amounts and counts are highly skewed. `np.log1p()` reduces their scale while keeping zeros valid. Three ratio/diversity columns are used without a log transform. We intentionally exclude `account_id`, the date and the target label.

This is plain pandas: create an empty DataFrame, loop through column names, and copy/transform columns.

In [3]:
log_columns = [
    "transaction_count", "total_sent", "total_received",
    "avg_transaction_amount", "max_transaction_amount", "unique_banks"
]
plain_columns = ["outgoing_ratio", "currency_diversity", "rapid_transaction_ratio"]

def make_features(rows):
    features = pd.DataFrame(index=rows.index)

    for column in log_columns:
        features["log_" + column] = np.log1p(rows[column])

    for column in plain_columns:
        features[column] = rows[column]

    assert np.isfinite(features.to_numpy()).all(), "Some features are missing or invalid."
    return features

X_train = make_features(train)
X_validation = make_features(validation)
X_test = make_features(test)

y_train = train["has_laundering_tx"]
y_validation = validation["has_laundering_tx"]
y_test = test["has_laundering_tx"]

print("Model feature count:", X_train.shape[1])
print("Features:", list(X_train.columns))

Model feature count: 9
Features: ['log_transaction_count', 'log_total_sent', 'log_total_received', 'log_avg_transaction_amount', 'log_max_transaction_amount', 'log_unique_banks', 'outgoing_ratio', 'currency_diversity', 'rapid_transaction_ratio']


## 4. Start with a simple baseline: logistic regression

**Why a baseline?** Before using a more complicated model, we should know what a familiar, simpler method can achieve.

Logistic regression combines feature values into a score. `StandardScaler` makes inputs comparable in scale; `class_weight="balanced"` adjusts for the low proportion of positive labels.

**PR-AUC (average precision)** summarizes how well a model puts rare positives near the top across thresholds. Higher is better. Here we measure it **on validation data**, not the held-out test.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

baseline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])

baseline.fit(X_train, y_train)
baseline_scores = baseline.predict_proba(X_validation)[:, 1]

baseline_pr_auc = average_precision_score(y_validation, baseline_scores)
print("Logistic regression validation PR-AUC:", round(baseline_pr_auc, 4))

Logistic regression validation PR-AUC: 0.0378


## 5. Train XGBoost with a frozen configuration

**Why XGBoost?** Decision-tree models can learn interactions and nonlinear patterns that a linear model may miss. XGBoost combines many small trees into one ranking model.

The parameter values below are **the documented, frozen configuration** used for this case study. We do not change the model to chase test results. `n_jobs=2` and `random_state=42` help reproduce the same behavior in the pinned environment.

`scale_pos_weight` is calculated **only from training labels**, so validation and test labels don't leak into model fitting.

In [5]:
from xgboost import XGBClassifier

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
class_weight = negative_count / positive_count

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=class_weight,
    random_state=42,
    n_jobs=2
)

model.fit(X_train, y_train)
xgb_scores = model.predict_proba(X_validation)[:, 1]

xgb_pr_auc = average_precision_score(y_validation, xgb_scores)
print("XGBoost validation PR-AUC:", round(xgb_pr_auc, 4))

XGBoost validation PR-AUC: 0.1286


## 6. Compare both methods under the same review workload

A higher PR-AUC is useful, but analysts also want a **decision-based metric**: how many positives would appear in the first **2,500 reviewed records**?

We sort each model's validation scores from highest to lowest and count positive labels in its first 2,500 rows. The same budget makes the comparison fair. The table below shows both statistical discrimination and potential queue quality.

**Important:** These are *validation-period measurements* newly reproduced from the frozen source. They are distinct from the previously published XGBoost **test-period** results.

In [6]:
review_budget = 2500
validation_labels = y_validation.to_numpy()

def count_positives_at_budget(scores, labels, budget):
    ranked_positions = np.argsort(-scores, kind="stable")
    top_labels = labels[ranked_positions[:budget]]
    return int(top_labels.sum())

baseline_found = count_positives_at_budget(baseline_scores, validation_labels, review_budget)
xgb_found = count_positives_at_budget(xgb_scores, validation_labels, review_budget)

comparison = pd.DataFrame({
    "Model": ["Logistic regression", "XGBoost"],
    "Validation PR-AUC": [baseline_pr_auc, xgb_pr_auc],
    "Validation ROC-AUC": [roc_auc_score(y_validation, baseline_scores),
                           roc_auc_score(y_validation, xgb_scores)],
    "Positives in first 2,500": [baseline_found, xgb_found],
    "Precision at 2,500": [baseline_found / review_budget, xgb_found / review_budget]
})

print(comparison.round(4).to_string(index=False))
assert baseline_found == 170 and xgb_found == 383

              Model  Validation PR-AUC  Validation ROC-AUC  Positives in first 2,500  Precision at 2,500
Logistic regression             0.0378              0.7995                       170              0.0680
            XGBoost             0.1286              0.8446                       383              0.1532


**What the comparison says:** XGBoost ranked **383** synthetic positives in the first 2,500 validation rows, versus **170** for logistic regression. Its validation PR-AUC was **0.1286** versus **0.0378**. This supports the use of XGBoost **for this frozen synthetic benchmark**, not a general claim that XGBoost will outperform other approaches in real AML systems.

The model choice is discussed transparently. These newly recorded validation figures should not be described as evidence of a previously run, exhaustive hyperparameter search.

## 7. Evaluate the frozen model on the later test period

Only now do we evaluate the held-out dates. The **2,500-row validation workload** is converted to the same fraction of the larger test dataset, yielding **2,747 review slots**.

We select the top-scored test rows and calculate:

- **Precision:** selected positives ÷ reviewed records.
- **Recall:** selected positives ÷ all positives in the test set.
- **Lift:** selected precision ÷ the positive-label rate in the full test set.

These metrics measure synthetic label ranking—not real investigations or prevented losses.

In [7]:
test_scores = model.predict_proba(X_test)[:, 1]
review_slots = round(len(test) * review_budget / len(validation))

test_labels = y_test.to_numpy()
selected_positives = count_positives_at_budget(test_scores, test_labels, review_slots)
precision = selected_positives / review_slots
recall = selected_positives / int(y_test.sum())
prevalence = float(y_test.mean())
lift = precision / prevalence

results = pd.DataFrame({
    "Metric": ["Test account-days", "Review slots", "Positive labels selected",
               "Precision", "Recall", "Lift", "ROC-AUC", "PR-AUC"],
    "Value": [len(test), review_slots, selected_positives,
              round(precision, 4), round(recall, 4), round(lift, 2),
              round(roc_auc_score(y_test, test_scores), 4),
              round(average_precision_score(y_test, test_scores), 4)]
})

print(results.to_string(index=False))
assert (review_slots, selected_positives) == (2747, 389)

                  Metric       Value
       Test account-days 494707.0000
            Review slots   2747.0000
Positive labels selected    389.0000
               Precision      0.1416
                  Recall      0.2595
                    Lift     46.7300
                 ROC-AUC      0.8523
                  PR-AUC      0.1214


### Interpretation

At the fixed review rate, the model places **389 synthetic positive labels** among **2,747** reviewed account-days out of **494,707** held-out rows. That is **14.16% precision**, **25.95% recall**, and **46.73× lift** compared with random review of the same number of account-days.

![Published capacity curve and test-period outcome](../images/AML_Canonical_Dashboard.png)

The [interactive version](https://mohammedmia1.github.io/AML_Data_Science_Project/) lets you change the review budget. Its underlying 15,000-point curve is built from the frozen ranking, not fabricated chart data.

## 8. MLOps explained for a data analyst

**MLOps** means *Machine Learning Operations*. Think of it as the practices that help a model stay **organized, reproducible, testable and maintainable** after someone first builds it. It's similar to keeping reliable SQL queries, documented dashboards and repeatable data pipelines—but for machine-learning models.

| Familiar analytics idea | MLOps equivalent | In this project |
|---|---|---|
| Save the exact source extract | **Data versioning / lineage** | Input TSV has a SHA-256 fingerprint, so a different dataset triggers an error. |
| Keep the logic in version control | **Code versioning** | GitHub preserves training scripts, notebook and changes over time. |
| Record filter logic and calculation assumptions | **Reproducible training** | Pinned Python packages, fixed dates, nine features, fixed model configuration and seed. |
| Save a dashboard or scheduled output | **Model artifacts** | XGBoost model, scored CSV, metrics JSON and capacity curve are generated together. |
| Run automated data-quality checks | **Continuous Integration (CI)** | GitHub Actions checks the published metrics, curve, script syntax and notebook structure. |
| Monitor whether reports become stale | **Production monitoring** | **Not implemented**; this is not a live AML monitoring system. |
| Publish a data service for users | **Model serving / deployment** | **Not implemented**; the website explores already-scored synthetic predictions, not live inference. |

### What happens when a file or model changes?

1. A developer changes Python code or replaces input data.
2. The frozen input checksum, split checks and artifact tests help catch unintended differences.
3. GitHub Actions performs quick checks whenever code is pushed.
4. A full model rerun requires the large, exact input TSV and is performed separately.

**Key distinction:** CI passing does **not** mean the model was retrained or validated on real banking data. The project demonstrates a small reproducibility workflow, **not** a production MLOps platform with orchestration, model registry, live serving, drift alerts or automatic retraining.

## 9. Reproducing the published files

For a *full rerun*, use the [canonical training script](../src/run_canonical.py). It writes the model, full scored export, metrics JSON and website capacity JSON to `artifacts/`.

For a *quick verification* without the large TSV, run `python tests/check_artifacts.py` from the repository root. This checks the published metrics and chart-data contract, but does not refit XGBoost.

The notebook outputs above come from executing the analysis against the exact checksum-verified synthetic TSV. They are evidence of this notebook run, not evidence that the SQL raw-data extraction was freshly repeated.

## 10. Limitations and suggested next experiments

- These are **synthetic benchmark labels**, not confirmed money laundering.
- The held-out test covers only **two days**; additional temporal holdouts are needed to assess stability.
- The account-day granularity means one account can appear on different days.
- Scores rank records but have not been proven to be calibrated probabilities.
- The SQL source-to-TSV reconstruction has been reviewed but has **not** been independently reconciled against raw transactions.
- No investigator time, alert disposition, cost savings, live deployment or model monitoring were measured.

**Next responsible steps:** Compare additional temporal holdouts, independently verify the SQL export against the frozen TSV, and evaluate workload/cost trade-offs using actual operational definitions if suitable data becomes available.

[Back to project README](../README.md) · [Technical guide](../PROJECT_GUIDE.md) · [Live case study](https://mohammedmia1.github.io/AML_Data_Science_Project/)